<a href="https://colab.research.google.com/github/aasth321/face_mask_detection/blob/main/training_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import kagglehub
import os

print("Downloading clean classification dataset...")
# Swapping to a pre-cropped classification folder structure: [with_mask, without_mask]
path = kagglehub.dataset_download("omkargurav/face-mask-dataset")

# The downloaded directory contains a nested folder named 'data'
DATA_DIR = os.path.join(path, "data")

# 1. Dataset Preprocessing & Augmentation Pipeline
# Added horizontal flipping to help model learn profile features better
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    horizontal_flip=True
)

train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

print(f"\nTarget Class Mapping Identified: {train_generator.class_indices}\n")

# 2. Optimized CNN Architecture
# Swapped Flatten() for GlobalAveragePooling2D() to reduce parameters from 7M down to 128K!
# This completely prevents dead weight gradient loops.
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),

    # Drops parameter size drastically while keeping essential spatial vectors
    GlobalAveragePooling2D(),

    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(2, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 3. Train Model
print("Training initialized...")
model.fit(train_generator, validation_data=val_generator, epochs=7)

# 4. Save clean weights structure
model.save('mask_detector.h5')
print("\nSuccess! A fresh 'mask_detector.h5' file has been saved to your workspace.")

Using Colab cache for faster access to the 'face-mask-dataset' dataset.
Found 6043 images belonging to 2 classes.
Found 1510 images belonging to 2 classes.

Target Class Mapping Identified: {'with_mask': 0, 'without_mask': 1}



/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training initialized...
Epoch 1/7
 51/189 ━━━━━━━━━━━━━━━━━━━━ 23s 167ms/step - accuracy: 0.5805 - loss: 0.6677

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


189/189 ━━━━━━━━━━━━━━━━━━━━ 56s 260ms/step - accuracy: 0.7140 - loss: 0.5570 - val_accuracy: 0.7219 - val_loss: 0.5642
Epoch 2/7
189/189 ━━━━━━━━━━━━━━━━━━━━ 19s 103ms/step - accuracy: 0.7554 - loss: 0.5113 - val_accuracy: 0.7656 - val_loss: 0.5146
Epoch 3/7
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 97ms/step - accuracy: 0.7723 - loss: 0.4743 - val_accuracy: 0.8172 - val_loss: 0.4210
Epoch 4/7
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 95ms/step - accuracy: 0.7951 - loss: 0.4412 - val_accuracy: 0.8079 - val_loss: 0.4722
Epoch 5/7
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 94ms/step - accuracy: 0.8256 - loss: 0.3976 - val_accuracy: 0.8636 - val_loss: 0.3113
Epoch 6/7
189/189 ━━━━━━━━━━━━━━━━━━━━ 19s 98ms/step - accuracy: 0.8749 - loss: 0.3034 - val_accuracy: 0.8940 - val_loss: 0.3011
Epoch 7/7
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 92ms/step - accuracy: 0.8994 - loss: 0.2660 - val_accuracy: 0.9026 - val_loss: 0.2721



Success! A fresh 'mask_detector.h5' file has been saved to your workspace.
